# **Evaluation Notebook**

### **Wikiart Project**

**Group x:**\
**Afonso Hermenegildo** - 202 | **Lara Santos** - 20221823 | **Marco Martins** - 20221951 | **André Nicolau** - 2022

Github repository: https://github.com/MarcoAFMartins/Wikiart_Project

---

# Table of Contents

1. [Classification report (accuracy, macro-F1, per-class precision/recall)](#section-1)  
2. [Confusion matrix (seaborn heatmap)](#section-2)  
3. [Training curves (loss & accuracy)](#section-3)  
4. [Grad-CAM visualisation](#section-4)  
5. [Misclassified samples analysis](#section-5)
6. [Model comparison table](#section-5)

---

# Imports

In [4]:
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import keras
from pathlib import Path
from sklearn.metrics import confusion_matrix, classification_report
from keras.utils import image_dataset_from_directory
from notebooks.baseline_cnn import BaselineModel

# Paths — adjust if needed
DATA_DIR    = Path('Data')
OUTPUTS_DIR = Path('outputs')
FIGURES_DIR = OUTPUTS_DIR / 'figures'
MODELS_DIR  = OUTPUTS_DIR / 'models'
NOTEBOOK_DIR = Path('notebooks')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE  = 32
SEED        = 42

ModuleNotFoundError: No module named 'notebooks.baseline_cnn'

# Load dataset

In [ ]:
train_df = image_dataset_from_directory(
    DATA_DIR / "Train",
    label_mode="categorical",
    image_size=(128, 128),
    batch_size=BATCH_SIZE,
    seed=SEED,
    shuffle=True,
)

val_df = image_dataset_from_directory(
    DATA_DIR / "Validation",
    label_mode="categorical",
    image_size=(128, 128),
    batch_size=BATCH_SIZE,
    seed=SEED,
    shuffle=False,
)

test_df = image_dataset_from_directory(
    DATA_DIR / "Test",
    label_mode="categorical",
    image_size=(128, 128),
    batch_size=BATCH_SIZE,
    seed=SEED,
    shuffle=False,
)

class_names = train_df.class_names
print(f"Classes: {len(class_names)} — {class_names}")

Found 9338 files belonging to 23 classes.
Found 2001 files belonging to 23 classes.
Found 2001 files belonging to 23 classes.
Classes: 23 — ['Albrecht_Durer', 'Boris_Kustodiev', 'Camille_Pissarro', 'Childe_Hassam', 'Claude_Monet', 'Edgar_Degas', 'Eugene_Boudin', 'Gustave_Dore', 'Ilya_Repin', 'Ivan_Aivazovsky', 'Ivan_Shishkin', 'John_Singer_Sargent', 'Marc_Chagall', 'Martiros_Saryan', 'Nicholas_Roerich', 'Pablo_Picasso', 'Paul_Cezanne', 'Pierre_Auguste_Renoir', 'Pyotr_Konchalovsky', 'Raphael_Kirchner', 'Rembrandt', 'Salvador_Dali', 'Vincent_van_Gogh']


In [7]:
# Load the best checkpoint saved during training
# Update the path to match whichever model you want to evaluate
model = BaselineModel()
model.build((None, 128, 128, 3))
model.load_weights("outputs/modules/baseline_cnn.keras/model.weights.h5")


print(f'Model loaded from: {model_path}')
model.summary()

NameError: name 'BaselineModel' is not defined

# Make Predictions

In [ ]:
y_true, y_pred, y_prob = [], [], []

for images, labels in test_df:
    probs = model.predict(images, verbose=0)
    y_prob.extend(probs)
    y_pred.extend(np.argmax(probs, axis=1))
    y_true.extend(np.argmax(labels.numpy(), axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_prob = np.array(y_prob)

accuracy = np.mean(y_true == y_pred)
print(f'Test accuracy: {accuracy:.4f}')

# Classification Report

## 4. Classification report
> Primary metric for this project is **macro-F1** because classes may be imbalanced.
> Macro-F1 gives equal weight to all classes regardless of size.
> If macro-F1 is much lower than accuracy, the model is biased toward majority classes.

In [ ]:
report = classification_report(y_true, y_pred, target_names=class_names)
print(report)

# Save to file for the report
with open(OUTPUTS_DIR / 'classification_report.txt', 'w') as f:
    f.write(report)
print('Saved to outputs/classification_report.txt')